# 05b · Life of a write

## The engineering question
When is a write visible, when is it indexed, and what survives a restart?

**Learning outcomes:** distinguish acknowledgment, visibility, persistence, and indexing; inspect segment/index state; explain logical deletion versus storage reclamation; limit conclusions from an orderly restart.

Qdrant persists updates and organizes points in segments. Background optimization can build indexes or merge segments while queries continue. A successful write response and a healthy server do not imply that an HNSW index is ready. We use synthetic vectors here to isolate storage behavior; no semantic retrieval claim follows from them.

**Operational effect:** this notebook restarts only the course's Qdrant Compose service to check persistence. Do not run other course queries concurrently with that cell. It does not stop unrelated Docker projects.

In [ ]:
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
from vector_masterclass.lesson import LessonRun
from vector_masterclass.config import root

plt.rcParams.update(
    {"figure.figsize": (8, 4), "axes.spines.top": False, "axes.spines.right": False}
)
lesson = LessonRun("05b", "B", docker=True)
print("Kernel:", sys.executable)
print("Workspace:", root())

In [ ]:
import time
import subprocess
from uuid import uuid4
from qdrant_client import QdrantClient, models as qm
from vector_masterclass.config import qdrant_url
from vector_masterclass.preflight import wait_ready

client = QdrantClient(url=qdrant_url(), timeout=20)
name = "vmc_05b_" + uuid4().hex[:8]
rng = np.random.default_rng(17)
vectors = rng.normal(size=(3000, 32)).astype("float32")
vectors /= np.linalg.norm(vectors, axis=1, keepdims=True)
rows = []
client.create_collection(
    name,
    vectors_config=qm.VectorParams(size=32, distance=qm.Distance.COSINE),
    optimizers_config=qm.OptimizersConfigDiff(indexing_threshold=64, default_segment_number=1),
)


def observe(stage):
    info = client.get_collection(name)
    rows.append(
        {
            "stage": stage,
            "visible_count": client.count(name, exact=True).count,
            "indexed_vectors": info.indexed_vectors_count or 0,
            "segments": info.segments_count,
            "status": str(info.status),
        }
    )


try:
    for start in range(0, 3000, 500):
        client.upsert(
            name,
            points=[
                qm.PointStruct(id=i, vector=vectors[i].tolist(), payload={"revision": 1})
                for i in range(start, start + 500)
            ],
            wait=True,
        )
    observe("write acknowledged")
    deadline = time.monotonic() + 45
    while time.monotonic() < deadline:
        info = client.get_collection(name)
        if info.status == qm.CollectionStatus.GREEN and (info.indexed_vectors_count or 0) >= 2900:
            break
        time.sleep(0.25)
    observe("after bounded optimizer wait")
    client.set_payload(name, payload={"revision": 2}, points=[200], wait=True)
    assert client.retrieve(name, ids=[200])[0].payload["revision"] == 2
    client.delete(name, points_selector=qm.PointIdsList(points=list(range(100))), wait=True)
    observe("after logical deletion")
    client.close()
    subprocess.run(
        ["docker", "compose", "restart", "qdrant"],
        cwd=root(),
        check=True,
        capture_output=True,
        text=True,
        timeout=40,
    )
    wait_ready(30)
    client = QdrantClient(url=qdrant_url(), timeout=20)
    observe("after orderly restart")
    retained = client.retrieve(name, ids=[200])[0].payload["revision"] == 2
    deleted_stays_absent = not client.retrieve(name, ids=[0])
finally:
    if client.collection_exists(name):
        client.delete_collection(name)
    client.close()
results = lesson.table(pd.DataFrame(rows))
display(results)
fig, ax = plt.subplots(figsize=(10, 4))
results.set_index("stage")[["visible_count", "indexed_vectors"]].rename(
    columns={"visible_count": "Visible points", "indexed_vectors": "Indexed vectors"}
).plot.bar(ax=ax)
ax.set(ylabel="Points/vectors", title="Visibility and indexing are separate observations")
ax.tick_params(axis="x", rotation=15)
ax.legend(loc="upper left", bbox_to_anchor=(1, 1))
fig.tight_layout()
lesson.figure(fig)
plt.show()

## Interpretation and exercise
Background work may finish before the first observation on this machine. If the visible and indexed counts already agree, report that observation rather than inventing an indexing lag. Deletion removes a point from results before disk compaction necessarily reclaims its bytes. Segment counts and index counts can change as optimization runs.

**Exercise:** increase the ingestion batch and record more frequent observations. Explain which conclusions change if the process is killed rather than stopped cleanly. Do not claim a power-loss durability guarantee from this orderly restart.

Production recovery requires documented write semantics, failure-specific tests, backups, and reconciliation with the source store. The next lesson addresses the missing transaction between Qdrant and an ingestion manifest. Use an exact scan for an oracle while data is quiescent; concurrent mutation changes the evaluation target.

In [ ]:
manifest = lesson.finish(
    {
        "all_writes_visible": int(results.iloc[0].visible_count) == 3000,
        "deletes_persisted": int(results.iloc[-1].visible_count) == 2900 and deleted_stays_absent,
        "update_persisted": retained,
    },
    observations={
        "storage_timeline": rows,
        "restart_kind": "orderly process restart; no power-loss claim",
    },
)

## References and next steps
Consult the [Qdrant documentation](https://qdrant.tech/documentation/),
the [MiniLM model card](https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2),
and [uv's Jupyter guide](https://docs.astral.sh/uv/guides/integration/jupyter/).
The exact pinned configuration is in `configs/models.json`; worked exercises are in `docs/SOLUTIONS.md`.
This notebook's measurements apply to the recorded local run, not a production deployment.